In [2]:
# ================================================
# 用 OpenAI gpt-5-nano 直接評估 RAG
# ================================================

!pip install -q "openai>=1.35.0" pandas tqdm

import os, json, re, shutil, time, getpass
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from google.colab import files
from openai import OpenAI

# ----------------------------
# 手動輸入 OpenAI API Key（強制每次重輸入）
# ----------------------------
API_KEY = getpass.getpass("請輸入你的 OPENAI_API_KEY（每次都需輸入新的 Key）: ")


# 設定成新的 key
os.environ["OPENAI_API_KEY"] = API_KEY

# 建立 openai client
client = OpenAI(api_key=API_KEY)
llm_model = "gpt-5-nano"

# ----------------------------
# 建立資料夾
# ----------------------------
BASE = Path("./eval_data")
GEN_DIR = BASE / "generated"
OUT = Path("./eval_out")

BASE.mkdir(parents=True, exist_ok=True)
GEN_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 上傳兩個檔案
# ----------------------------
print("請上傳 generated_results.csv")
uploaded = files.upload()
gen_local = next(iter(uploaded.keys()))
gen_path = GEN_DIR / "generated_results.csv"
shutil.move(gen_local, gen_path)

print("請上傳 gold_labels.csv")
uploaded = files.upload()
gold_local = next(iter(uploaded.keys()))
gold_path = BASE / "gold_labels.csv"
shutil.move(gold_local, gold_path)

# ----------------------------
# 讀 generated_results.csv
# ----------------------------
gen = pd.read_csv(gen_path)

need_cols = {"qid", "query", "response", "contexts"}
miss = need_cols - set(gen.columns)
if miss:
    raise ValueError(f"generated_results.csv 缺少欄位：{miss}")


def parse_contexts(x):
    """把 contexts 字串還原成 list[str]"""
    try:
        arr = json.loads(x)
        out = []
        if isinstance(arr, list):
            for it in arr:
                if isinstance(it, dict):
                    out.append(str(it.get("text", "")))
                else:
                    out.append(str(it))
            return out
    except:
        pass
    return [str(x)]

gen["contexts_list"] = gen["contexts"].apply(parse_contexts)

# ----------------------------
# 讀 gold_labels.csv
# ----------------------------
gold = pd.read_csv(gold_path)

if not {"query", "suppose_answer"}.issubset(gold.columns):
    raise ValueError("gold_labels.csv 必須包含 query, suppose_answer 欄位")


def ynf_to_ref(row):
    q = row["query"]
    label = str(row["suppose_answer"]).strip().upper()
    if label == "Y":
        return f"問題「{q}」的正確答案是：是。"
    elif label == "N":
        return f"問題「{q}」的正確答案是：否。"
    else:
        return f"問題「{q}」沒有足夠資料回答，應回答查無相關資料。"

gold_min = gold[["query", "suppose_answer"]].copy()
gold_min["ground_truth"] = gold_min.apply(ynf_to_ref, axis=1)

gen = gen.merge(gold_min, on="query", how="left")

# ----------------------------
# 規則版 Y/N/F
# ----------------------------
Y_PAT = r"(是|可以|會|有|屬於|為|用於)"
N_PAT = r"(不是|不可|不會|沒有|無法|非)"

def heuristic_label(a: str):
    s = (a or "").strip()
    if not s:
        return "F"
    if re.search(N_PAT, s):
        return "N"
    if re.search(Y_PAT, s):
        return "Y"
    if "無相關資料" in s or "沒有資料" in s:
        return "F"
    return "F"

gen["pred_label_from_resp"] = gen["response"].astype(str).apply(heuristic_label)
gen["gold_label"] = gen["suppose_answer"]

# ----------------------------
# LLM 評分函式
# ----------------------------
def score_one_with_llm(question, answer, ctx_text, ref_text):
    prompt = f"""
請用 JSON 回覆以下四項評分（數值 0~1）：

{{
  "faithfulness": 0.0,
  "answer_relevancy": 0.0,
  "context_precision": 0.0,
  "context_recall": 0.0
}}

---

問題：
{question}

模型回答：
{answer}

檢索到的 context：
{ctx_text}

理想答案（若無則代表空）：
{ref_text}
"""
    try:
        resp = client.chat.completions.create(
            model=llm_model,
            messages=[
                {"role": "system", "content": "你是 RAG 評估助手，請精準輸出 JSON。"},
                {"role": "user", "content": prompt},
            ],
        )

        text = resp.choices[0].message.content.strip()
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            raise ValueError("未找到 JSON")

        data = json.loads(m.group(0))

        def norm(x):
            try:
                x = float(x)
                return min(max(x, 0.0), 1.0)
            except:
                return 0.0

        return {
            "faithfulness": norm(data.get("faithfulness", 0)),
            "answer_relevancy": norm(data.get("answer_relevancy", 0)),
            "context_precision": norm(data.get("context_precision", 0)),
            "context_recall": norm(data.get("context_recall", 0)),
        }

    except Exception as e:
        print(f"[WARN] 打分失敗：{e}")
        return {"faithfulness": 0, "answer_relevancy": 0, "context_precision": 0, "context_recall": 0}

# ----------------------------
# 評分系統
# ----------------------------
START = 0
END = 50
subset = gen.iloc[START:END]

print(f"\n只評分第 {START} ~ {END-1} 題（共 {len(subset)} 題）\n")

q_series = subset["query"].astype(str)
a_series = subset["response"].astype(str)
ctx_series = subset["contexts_list"].apply(lambda xs: " ".join(xs))
gt_series = subset["ground_truth"].astype(str)

llm_scores = []
RATE_SLEEP = 2.0

for i, (q, a, ctx, gt) in enumerate(zip(q_series, a_series, ctx_series, gt_series)):
    print(f"Scoring question {START+i} …")
    s = score_one_with_llm(q, a, ctx, gt)
    llm_scores.append(s)
    time.sleep(RATE_SLEEP)



df_scores_llm = pd.DataFrame(llm_scores)

# ----------------------------
# 輸出 CSV
# ----------------------------
df_scores_llm.to_csv(OUT / "rag_scores_llm.csv", index=False, encoding="utf-8-sig")

per_q = pd.concat(
    [
        subset[["qid", "query", "gold_label", "pred_label_from_resp"]].reset_index(drop=True),
        df_scores_llm.reset_index(drop=True),
    ],
    axis=1,
)

per_q.to_csv(OUT / "rag_scores_llm_per_question.csv", index=False, encoding="utf-8-sig")

print("\n完成！")
print("輸出：")
print(" - eval_out/rag_scores_llm.csv")
print(" - eval_out/rag_scores_llm_per_question.csv")

請輸入你的 OPENAI_API_KEY（每次都需輸入新的 Key）: ··········
請上傳 generated_results.csv


Saving generated_results.csv to generated_results.csv
請上傳 gold_labels.csv


Saving gold_labels.csv to gold_labels.csv

只評分第 0 ~ 49 題（共 50 題）

Scoring question 0 …
Scoring question 1 …
Scoring question 2 …
Scoring question 3 …
Scoring question 4 …
Scoring question 5 …
Scoring question 6 …
Scoring question 7 …
Scoring question 8 …
Scoring question 9 …
Scoring question 10 …
Scoring question 11 …
Scoring question 12 …
Scoring question 13 …
Scoring question 14 …
Scoring question 15 …
Scoring question 16 …
Scoring question 17 …
Scoring question 18 …
Scoring question 19 …
Scoring question 20 …
Scoring question 21 …
Scoring question 22 …
Scoring question 23 …
Scoring question 24 …
Scoring question 25 …
Scoring question 26 …
Scoring question 27 …
Scoring question 28 …
Scoring question 29 …
Scoring question 30 …
Scoring question 31 …
Scoring question 32 …
Scoring question 33 …
Scoring question 34 …
Scoring question 35 …
Scoring question 36 …
Scoring question 37 …
Scoring question 38 …
Scoring question 39 …
Scoring question 40 …
Scoring question 41 …
Scoring question 42 

In [6]:
from google.colab import files

files.download("eval_out/rag_scores_llm.csv")
files.download("eval_out/rag_scores_llm_per_question.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>